# 🔔 RUNG CHUÔNG VÀNG — TIKTOK LIVE INTERACTIVE GAME 9:16
### HỆ THỐNG PHÁT SÓNG GAMESHOW TRÍ TUỆ TƯƠNG TÁC TRÊN GOOGLE COLAB

Hệ thống phát sóng tương tác chuẩn khung hình dọc **9:16 (1080x1920)** mô phỏng sàn đấu 100 thí sinh Rung Chuông Vàng, hỗ trợ:
- **Mode A (Khuyên dùng - Nhẹ máy nhất):** Colab làm máy chủ -> Tạo đường link HTTPS Cloudflare Tunnel để dán vào mục **Liên kết (Browser Source)** trong **TikTok LIVE Studio** hoặc **OBS Studio** (kích thước 1080 x 1920).
- **Mode B (Tự động 24/7 - Rảnh tay):** Colab tự tạo Màn hình ảo (Xvfb 1080x1920) và dùng FFmpeg đẩy luồng trực tiếp lên TikTok bằng **Stream Key** mà không cần mở máy tính cá nhân!

In [ ]:
# @title 🚀 MODE A: Khởi Chạy Server & Lấy Link Dán Vào TikTok LIVE Studio / OBS { display-mode: "form" }
# @markdown Điền tên tài khoản TikTok (ví dụ: `haisound` hoặc `@haisound`) rồi bấm nút **Play ▶** bên trái:

TikTok_Username = "haisound" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/4] Đang chuẩn bị mã nguồn Game Rung Chuông Vàng...")
# Nếu chưa có thì clone về, nếu có rồi thì cập nhật
!git clone https://github.com/tuna1710/tiktok-live-game-skill.git /content/app 2>/dev/null || (cd /content/app && git fetch origin && git reset --hard origin/main && git pull 2>/dev/null || true)

# Kiểm tra thư mục game
game_dir = "/content/app/games/rung-chuong-vang"
if not os.path.exists(game_dir):
    game_dir = "/content/rung-chuong-vang"

os.chdir(game_dir) if os.path.exists(game_dir) else None

print("⏳ [2/4] Đang cài đặt thư viện Node.js & Cloudflare Tunnel...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("⏳ [3/4] Đang dọn dẹp tiến trình cũ và khởi chạy Game Server...")
!pkill -9 -f "server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

clean_username = TikTok_Username.replace('@', '').strip()
env = os.environ.copy()
env["PORT"] = "3000"
env["TIKTOK_USERNAME"] = clean_username

with open("/tmp/game_server.log", "w") as f_log:
    server_proc = subprocess.Popen(["node", "server.js"], cwd=game_dir, env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

print("⏳ [4/4] Đang tạo đường truyền HTTPS công khai qua Cloudflare Tunnel...")
with open("/tmp/cloudflared.log", "w") as cf_log:
    tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    obs_link = f"{public_url}/index.html"
    admin_link = f"{public_url}/admin.html"
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #1f233a 0%, #0c0e18 100%); border: 2px solid #ffd700; border-radius: 16px; padding: 22px; color: #ffffff; font-family: sans-serif; box-shadow: 0 8px 30px rgba(255,215,0,0.3);">
        <h2 style="color: #ffd700; margin-top: 0; font-size: 24px;">🔔 GAME RUNG CHUÔNG VÀNG ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e0e6ed;">
            Đường link công khai HTTPS đã được thiết lập qua Cloudflare Tunnel.<br>
            Tài khoản TikTok Live đang lắng nghe: <b style="color: #00f2fe;">@{clean_username}</b>
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{obs_link}" target="_blank" style="background: linear-gradient(135deg, #ffd700 0%, #ff8c00 100%); color: #000; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; box-shadow: 0 4px 15px rgba(255,215,0,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME (Dán link này vào OBS / TikTok Studio)
            </a>
            <a href="{admin_link}" target="_blank" style="background: linear-gradient(135deg, #00b894 0%, #00cec9 100%); color: #000; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px;">
                🎛️ [2] MỞ TRANG ĐIỀU KHIỂN & GIẢ LẬP (MOCK PANEL)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.6); border-left: 4px solid #ffd700; padding: 12px 14px; border-radius: 6px; font-size: 13.5px; color: #ffe4b5;">
            📌 <b>Link dán vào OBS Browser Source (Kích thước: Rộng 1080 x Cao 1920):</b><br>
            <code style="color: #00f2fe; background: #000; padding: 5px 10px; border-radius: 4px; display: inline-block; margin-top: 6px; font-size: 14px; word-break: break-all;">{obs_link}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Chưa lấy được link Cloudflare. Log chi tiết:")
    !cat /tmp/cloudflared.log


---

In [ ]:
# @title 🔴 MODE B: Phát Trực Tiếp Lên TikTok Bằng Stream Key (Không Cần OBS / Máy Tính) { display-mode: "form" }
# @markdown Lấy URL máy chủ và Khóa luồng (Stream Key) từ TikTok LIVE Studio -> Dán vào đây -> Bấm nút **Play ▶**:

Stream_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
Stream_Key = "" # @param {type:"string"}
TikTok_Username = "haisound" # @param {type:"string"}

!python3 live_ngam_tiktok.py --stream-url "$Stream_URL" --stream-key "$Stream_Key" --tiktok-username "$TikTok_Username"
